In [ ]:
!pip3 install -U sentence-transformers

# Part 1: Semantic Search

In [10]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from operator import itemgetter

def get_top_search_result(query, print_top_k: bool = False) -> str:
    minilm_model = SentenceTransformer('all-MiniLM-L6-v2')
    minilm_cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    sentences = [
        "A purple pig and a green donkey flew a kite in the middle of the night and ended up sunburnt.",
        "My pet purple pig disappeared in the morning.",
        "He didn't want to go to the dentist, yet he went anyway.",
        "He strives to keep the best lawn in the neighborhood.",
        "As he looked out the window, he saw a clown walk by.",
        "Rock music approaches at high velocity.",
        "Mom didn't understand why no one else wanted a hot tub full of jello.",
        "The tattered work gloves speak of the many hours of hard labor he endured throughout his life.",
        "The teens wondered what was kept in the red shed on the far edge of the school grounds.",
        "Don't put peanut butter on the dog's nose.",
        "The river stole the gods."
    ]

    doc_embeddings = minilm_model.encode(sentences)
    query_embedding = minilm_model.encode([query])
    k_elements_to_rerank = 3

    # First pass,
    first_pass_similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

    sorted_docs_by_first_pass = sorted(zip(first_pass_similarities, sentences), key=itemgetter(0), reverse=True)

    # Second pass, reranking with a cross encoder

    docs_and_similarities_after_second_pass = []
    for sim_doc_pair in sorted_docs_by_first_pass[:k_elements_to_rerank]:
        sentence = sim_doc_pair[1]
        reranked_sim = minilm_cross_encoder.predict((query, sentence))
        docs_and_similarities_after_second_pass.append((reranked_sim, sentence))

    sorted_docs_after_second_pass = sorted(docs_and_similarities_after_second_pass, key=itemgetter(0), reverse=True)

    if print_top_k:
        print("\n")
        print("TOP K SORTED DOCS AFTER FIRST PASS -----------------")
        print(sorted_docs_by_first_pass[:k_elements_to_rerank])
        print("\n")
        print("TOP K SORTED DOCS AFTER SECOND PASS -----------------")
        print(sorted_docs_after_second_pass)
    
    return sorted_docs_after_second_pass[0][1]


# Part 2: RAG

In [ ]:
!pip install accelerate

In [23]:
import torch
from transformers import pipeline

pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.bfloat16, device_map="auto")
query = "What happened to my pet purple pig?"
torch.manual_seed = 0
messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot. Speak in first person from the chatbot's perspective. Do not structure your responses like a letter.",
    },
    {   "role": "user",
        "content": "Generate a helpful and humanlike answer using the top search result for the user's query which is given here: " + get_top_search_result(query, True)
    },
]
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, return_full_text=False, no_repeat_ngram_size=2, max_new_tokens=256, do_sample=True, temperature=0.5, top_k=100, top_p=0.95)

print("--------------------RAG RESPONSE HERE--------------------")
print(outputs[0]["generated_text"])



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12601.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13623.77it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.c



TOP K SORTED DOCS AFTER FIRST PASS -----------------
[(np.float32(0.83414876), 'My pet purple pig disappeared in the morning.'), (np.float32(0.5071158), 'A purple pig and a green donkey flew a kite in the middle of the night and ended up sunburnt.'), (np.float32(0.26503634), "Don't put peanut butter on the dog's nose.")]


TOP K SORTED DOCS AFTER SECOND PASS -----------------
[(np.float32(6.3267655), 'My pet purple pig disappeared in the morning.'), (np.float32(2.261089), 'A purple pig and a green donkey flew a kite in the middle of the night and ended up sunburnt.'), (np.float32(-11.200651), "Don't put peanut butter on the dog's nose.")]
--------------------RAG RESPONSE HERE--------------------
Dear user,

It'd be a pleasure to help you with your concern regarding your beloved pet, Purple Pig. We understand how heartbroken you must be, and we want to provide you all the necessary information and support that you require.
 
Purple has been a constant companion to me since I was a kit

# Model Analysis

**_NOTE:_** To get a good result, you should use 2 models because models good at semantic search (i.e. they have a good encoder) usually aren't good at generating text (i.e. they have a bad decoder).


### What pretrained model did I use?
Semantic Search: I used `all-MiniLM-L6-v2` which is available in the `SentenceTransformers` package but is derived from the BERT architecture.\
RAG: I used `TinyLlama-1.1B-Chat-v1.0`.

### What was the original training objective?
Semantic Search: This model had a ["self-supervised contrastive learning objective"](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2#:~:text=self%2Dsupervised%20contrastive%20learning%20objective).\
RAG: TinyLlama 1.1B v1.0's training goal was to be a casual, conversational chat-based model.

### Was this model fine tuned? If yes, how?
Semantic Search: This model was fine tuned on a ["1B sentence pairs dataset"](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2#:~:text=fine%2Dtuned%20in%20on%20a%201B%20sentence%20pairs%20dataset). The training data consisted of pairs of related sentences. Then, the model would be given a sentence, and have to guess which sentence out of a set of possible sentences it was supposed to be paired with. They calculated this by calculating the cosine similarity between the sentence to pair and the possible sentences and then "applied the cross entropy loss by comparing with true pairs". \
RAG: TinyLlama 1.1 Chat v1.0 was fine tuned on `openbmb`'s UltraFeedback dataset - a series of prompts, their models, completions, and their correct and incorrect answers ranked by GPT-4.\
I did not do any additional fine-tuning on either model.

### Why is this model appropriate for your task?
Semantic Search: miniLM's encoder is very good for the semantic search task because BERT-style models are great for context-aware search queries due to their bidirectional nature. Thus, the embeddings they create are highly specialized for semantic searches, especially after reranking.\
RAG: I picked this transformer because it has an encoder component so it can actually generate text unlike `miniLM`, it was trained with having conservations in mind, and because it's lightweight enough to run on my laptop.


**_Final Thoughts:_** The reranking doesn't change the order of the search results in this particular dataset but it's still a good exercise